In [ ]:
# Fas 1: Datainsamling — Örebro Bostadsprisanalys
## 0. Installation och setup
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import json
import time
import random
import os
import re
from datetime import datetime
from tqdm.notebook import tqdm

print(f"Notebook startad: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Pandas version: {pd.__version__}")

Notebook startad: 2026-03-15 13:11
Pandas version: 3.0.1


In [38]:
# Skapa mappar
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/external', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Mappar skapade!')

Mappar skapade!


In [ ]:
### 1a. Metod A — Selenium Scraper
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


def create_driver():
    """Skapa en Chrome WebDriver med rätt inställningar."""
    options = Options()
    
    # Kör i headless mode (ingen synlig webbläsare)
    options.add_argument('--headless=new')
    
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--lang=sv-SE')
    options.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    )
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver


# Testa att drivern fungerar
try:
    driver = create_driver()
    driver.get('https://www.hemnet.se')
    print(f'Driver skapad! Sidtitel: {driver.title}')
    driver.quit()
    print('Selenium fungerar!')
except Exception as e:
    print(f'Selenium-fel: {e}')
    print('\nSe till att Chrome är installerat och kör:')
    print('  pip install selenium webdriver-manager')

Driver skapad! Sidtitel: Hemnet - Sveriges största bostadsplattform
Selenium fungerar!


In [ ]:
# HEMNET SELENIUM SCRAPER — Örebro kommun slutpriser

def accept_cookies(driver):
    """Klicka bort cookie-popup om den visas."""
    try:
        cookie_btn = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 
                '[data-testid="accept-cookies"], '
                'button[class*="consent"], '
                'button[class*="cookie"], '
                '#onetrust-accept-btn-handler'
            ))
        )
        cookie_btn.click()
        print('  Cookie-popup accepterad')
        time.sleep(1)
    except:
        pass  # Ingen cookie-popup


def scrape_sold_page(driver, url):
    """
    Scrapa en sida med slutpriser från Hemnet.
    
    Returnerar en lista med dicts, en per bostad.
    """
    listings = []
    
    driver.get(url)
    time.sleep(random.uniform(2.0, 4.0))  # Rate limiting
    
    # Hämta sidans HTML efter att JavaScript laddat klart
    soup = BeautifulSoup(driver.page_source, 'lxml')
    
    # Hemnet's slutpris-kort — testa flera selektorer
    # (de ändrar sina CSS-klasser ibland)
    cards = soup.select(
        '[data-testid="result-list"] a, '
        '.sold-property-listing, '
        'a[class*="sold-property"], '
        'a[href*="/salda/"]'
    )
    
    if not cards:
        # Fallback: hitta alla länkar som pekar på sålda bostäder
        cards = soup.find_all('a', href=re.compile(r'/salda/\w+/'))
    
    for card in cards:
        try:
            listing = parse_card(card)
            if listing and listing.get('slutpris'):
                listings.append(listing)
        except Exception as e:
            continue
    
    return listings


def parse_card(card):
    """
    Extrahera data från ett slutpris-kort.
    
    Hemnet's kort innehåller vanligtvis:
    - Adress (rubrik/länk)
    - Område, kommun
    - Slutpris
    - Boarea, antal rum
    - Avgift (bostadsrätt)
    - Sålddatum
    - Prisförändring (%)
    """
    text = card.get_text(' ', strip=True)
    
    if not text or len(text) < 10:
        return None
    
    listing = {
        'raw_text': text,
        'url': card.get('href', ''),
    }
    
    # Gör URL komplett om den är relativ
    if listing['url'] and not listing['url'].startswith('http'):
        listing['url'] = 'https://www.hemnet.se' + listing['url']
    
    # ---- PRIS ----
    price_match = re.search(r'(?:Slutpris\s+)?([\d\s]+)\s*kr', text)
    if price_match:
        price_str = price_match.group(1).replace(' ', '').replace('\xa0', '')
        try:
            listing['slutpris'] = int(price_str)
        except ValueError:
            pass
    
    # ---- BOAREA ----
    area_match = re.search(r'([\d,\.]+)(?:\+[\d,\.]+)?\s*m[²2]', text)
    if area_match:
        area_str = area_match.group(1).replace(',', '.')
        try:
            listing['boarea_kvm'] = float(area_str)
        except ValueError:
            pass
    
    # ---- ANTAL RUM ----
    rooms_match = re.search(r'([\d,]+)\s*rum', text)
    if rooms_match:
        rooms_str = rooms_match.group(1).replace(',', '.')
        try:
            listing['antal_rum'] = float(rooms_str)
        except ValueError:
            pass
    
    # ---- AVGIFT ----
    fee_match = re.search(r'([\d\s]+)\s*kr/m[åa]n', text)
    if fee_match:
        fee_str = fee_match.group(1).replace(' ', '').replace('\xa0', '')
        try:
            listing['avgift_kr'] = int(fee_str)
        except ValueError:
            pass
    
    # ---- DATUM ----
    date_match = re.search(
        r'[Ss]åld\s+(\d{1,2})\s+(jan|feb|mar|apr|maj|jun|jul|aug|sep|okt|nov|dec)\.?\s+(\d{4})',
        text, re.IGNORECASE
    )
    if date_match:
        months = {'jan':'01','feb':'02','mar':'03','apr':'04','maj':'05','jun':'06',
                  'jul':'07','aug':'08','sep':'09','okt':'10','nov':'11','dec':'12'}
        day = date_match.group(1).zfill(2)
        month = months.get(date_match.group(2).lower()[:3], '00')
        year = date_match.group(3)
        listing['sald_datum'] = f'{year}-{month}-{day}'
    
    # ---- PRISFÖRÄNDRING ----
    change_match = re.search(r'([+-]\d+)\s*%', text)
    if change_match:
        listing['prisforandring_pct'] = int(change_match.group(1))
    elif '±0' in text:
        listing['prisforandring_pct'] = 0
    
    # ---- KVM-PRIS ----
    kvm_match = re.search(r'([\d\s]+)\s*kr/m[²2]', text)
    if kvm_match:
        kvm_str = kvm_match.group(1).replace(' ', '').replace('\xa0', '')
        try:
            listing['pris_per_kvm'] = int(kvm_str)
        except ValueError:
            pass
    
    return listing if listing.get('slutpris') else None


print('Scraper-funktioner laddade!')

Scraper-funktioner laddade!


In [ ]:

# KONFIGURATION — Anpassa dessa efter behov

# Hemnet slutpris-URL:er för Örebro kommun

SCRAPE_URLS = {
    'lagenheter': 'https://www.hemnet.se/salda/lagenhet/orebro-kommun',
    'villor': 'https://www.hemnet.se/salda/villa/orebro-kommun',
    'radhus': 'https://www.hemnet.se/salda/radhus/orebro-kommun',
}

MAX_PAGES_PER_TYPE = 50  # Hemnet visar max 50 sidor per sökning

print('URL:er konfigurerade:')
for typ, url in SCRAPE_URLS.items():
    print(f'  {typ}: {url}')

URL:er konfigurerade:
  lagenheter: https://www.hemnet.se/salda/lagenhet/orebro-kommun
  villor: https://www.hemnet.se/salda/villa/orebro-kommun
  radhus: https://www.hemnet.se/salda/radhus/orebro-kommun


In [ ]:

# SCRAPING


all_listings = []
driver = create_driver()

try:
    for bostadstyp, base_url in SCRAPE_URLS.items():
        print(f'\n{"="*50}')
        print(f'  Scrapar: {bostadstyp.upper()}')
        print(f'{"="*50}')
        
        type_listings = []
        
        for page in tqdm(range(1, MAX_PAGES_PER_TYPE + 1), desc=bostadstyp):
            url = f'{base_url}?page={page}'
            
            # Första sidan: acceptera cookies
            if page == 1:
                driver.get(url)
                time.sleep(3)
                accept_cookies(driver)
            
            page_listings = scrape_sold_page(driver, url)
            
            if not page_listings:
                print(f'  Inga fler resultat på sida {page}. Klart!')
                break
            
            for listing in page_listings:
                listing['bostadstyp'] = bostadstyp
                listing['scrapad'] = datetime.now().isoformat()
            
            type_listings.extend(page_listings)
            
            # Visa progress var 10:e sida
            if page % 10 == 0:
                print(f'    Sida {page}: {len(type_listings)} listningar totalt')
        
        print(f'  Resultat {bostadstyp}: {len(type_listings)} listningar')
        all_listings.extend(type_listings)

finally:
    driver.quit()
    print(f'\nDriver stängd.')

print(f'\n{"="*50}')
print(f'TOTALT: {len(all_listings)} listningar scrapade')
print(f'{"="*50}')


  Scrapar: LAGENHETER


lagenheter:   0%|          | 0/50 [00:00<?, ?it/s]

    Sida 10: 500 listningar totalt
    Sida 20: 1000 listningar totalt
    Sida 30: 1500 listningar totalt
    Sida 40: 2000 listningar totalt
    Sida 50: 2500 listningar totalt
  Resultat lagenheter: 2500 listningar

  Scrapar: VILLOR


villor:   0%|          | 0/50 [00:00<?, ?it/s]

    Sida 10: 500 listningar totalt
    Sida 20: 1000 listningar totalt


In [ ]:
if all_listings:
    df_raw = pd.DataFrame(all_listings)

In [ ]:
# FIXAR PARSINGEN — korrekt slutpris, avgift, område

import re

def reparse_listing(raw_text, url='', bostadstyp=''):
    """Omtolka raw_text med förbättrad regex."""
    listing = {}
    text = raw_text
    
    # SLUTPRIS — måste komma efter ordet "Slutpris"
    price_match = re.search(r'Slutpris\s+([\d\s]+)\s*kr', text)
    if price_match:
        listing['slutpris'] = int(price_match.group(1).replace(' ', '').replace('\xa0', ''))
    
    # AVGIFT — siffror följt av "kr/mån"
    fee_match = re.search(r'([\d\s]+)\s*kr/mån', text)
    if fee_match:
        listing['avgift_kr'] = int(fee_match.group(1).replace(' ', '').replace('\xa0', ''))
    
    # BOAREA
    area_match = re.search(r'([\d,\.]+)(?:\+[\d,\.]+)?\s*m[²2]', text)
    if area_match:
        listing['boarea_kvm'] = float(area_match.group(1).replace(',', '.'))
    
    # ANTAL RUM
    rooms_match = re.search(r'([\d,]+)\s*rum', text)
    if rooms_match:
        listing['antal_rum'] = float(rooms_match.group(1).replace(',', '.'))
    
    # DATUM
    date_match = re.search(
        r'[Ss]åld\s+(\d{1,2})\s+(jan|feb|mar|apr|maj|jun|jul|aug|sep|okt|nov|dec)\.?\s+(\d{4})',
        text, re.IGNORECASE
    )
    if date_match:
        months = {'jan':'01','feb':'02','mar':'03','apr':'04','maj':'05','jun':'06',
                  'jul':'07','aug':'08','sep':'09','okt':'10','nov':'11','dec':'12'}
        day = date_match.group(1).zfill(2)
        month = months.get(date_match.group(2).lower()[:3], '00')
        year = date_match.group(3)
        listing['sald_datum'] = f'{year}-{month}-{day}'
    
    # PRISFÖRÄNDRING
    change_match = re.search(r'([+-]\d+)\s*%', text)
    if change_match:
        listing['prisforandring_pct'] = int(change_match.group(1))
    elif '±0' in text:
        listing['prisforandring_pct'] = 0
    
    # KVM-PRIS — siffror följt av "kr/m²"
    kvm_match = re.search(r'([\d\s]+)\s*kr/m[²2]', text)
    if kvm_match:
        listing['pris_per_kvm'] = int(kvm_match.group(1).replace(' ', '').replace('\xa0', ''))
    
    # OMRÅDE — första texten före "Örebro kommun"
    area_name = re.search(r'^[^,]*?([A-ZÅÄÖa-zåäö\s/]+),\s*Örebro', text)
    if area_name:
        listing['omrade'] = area_name.group(1).strip()
    
    # Behåll URL och bostadstyp
    listing['url'] = url
    listing['bostadstyp'] = bostadstyp
    listing['raw_text'] = raw_text
    
    return listing

# Applicera på all data
fixed_listings = []
for _, row in df_raw.iterrows():
    fixed = reparse_listing(
        row['raw_text'], 
        row.get('url', ''), 
        row.get('bostadstyp', '')
    )
    if fixed.get('slutpris'):
        fixed_listings.append(fixed)

df_fixed = pd.DataFrame(fixed_listings)

print(f'Fixade rader: {len(df_fixed)} av {len(df_raw)}')
print(f'\nSlutpris-statistik:')
print(f'  Min:    {df_fixed["slutpris"].min():>12,} kr')
print(f'  Median: {df_fixed["slutpris"].median():>12,.0f} kr')
print(f'  Max:    {df_fixed["slutpris"].max():>12,} kr')
print(f'\nFörsta 5 rader:')
display(df_fixed[['omrade','slutpris','boarea_kvm','antal_rum','avgift_kr','sald_datum','prisforandring_pct','bostadstyp']].head())

Fixade rader: 6622 av 6622

Slutpris-statistik:
  Min:         110,000 kr
  Median:    2,407,500 kr
  Max:      14,000,000 kr

Första 5 rader:


,omrade,slutpris,boarea_kvm,antal_rum,avgift_kr,sald_datum,prisforandring_pct,bostadstyp
0,Lägenhet Rynninge,2700000,117.0,4.0,8768.0,2026-03-13,8.0,lagenheter
1,Lägenhet Hovsta,1670000,84.0,3.0,5490.0,2026-03-11,1.0,lagenheter
2,Lägenhet Centralt Öster,1795000,44.0,1.0,2790.0,2026-03-11,-10.0,lagenheter
3,B Lägenhet Sörby,1380000,53.0,2.0,4906.0,2026-03-11,-1.0,lagenheter
4,Lägenhet Lundby,1000000,78.5,3.0,4603.0,2026-03-11,-9.0,lagenheter


In [ ]:
# Spara fixad data
df_fixed.to_csv('../data/raw/hemnet_orebro_fixed.csv', index=False, encoding='utf-8-sig')
print(f'Fixad data sparad: {len(df_fixed)} rader')

Fixad data sparad: 6622 rader


In [ ]:
SCB_BASE = 'https://api.scb.se/OV0104/v1/doris/sv/ssd'

In [ ]:
# SCB API — Grundfunktioner

SCB_BASE = 'https://api.scb.se/OV0104/v1/doris/sv/ssd'
OREBRO_KOMMUN = '1880'  # SCB:s kod för Örebro kommun


def scb_get(table_path):
    """Hämta metadata om en SCB-tabell (vilka variabler som finns)."""
    url = f'{SCB_BASE}/{table_path}'
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    return r.json()


def scb_post(table_path, query):
    """Hämta data från en SCB-tabell med en query."""
    url = f'{SCB_BASE}/{table_path}'
    r = requests.post(url, json=query, timeout=30)
    r.raise_for_status()
    data = r.json()
    
    # Konvertera SCB:s JSON-stat till DataFrame
    cols = [c['text'] for c in data.get('columns', [])]
    rows = [item['key'] + item['values'] for item in data.get('data', [])]
    
    return pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame()


# Testa att API:et fungerar
try:
    top = scb_get('')
    print('SCB API fungerar!')
    print(f'Antal ämnesområden: {len(top)}')
    for item in top[:5]:
        print(f'  {item["id"]}: {item["text"]}')
except Exception as e:
    print(f'SCB API-fel: {e}')

SCB API fungerar!
Antal ämnesområden: 22
  AA: Ämnesövergripande statistik
  AM: Arbetsmarknad
  BE: Befolkning
  BO: Boende, byggande och bebyggelse
  EN: Energi


In [ ]:
# 2a. MEDELINKOMST per år — Örebro kommun

print('Hämtar inkomstdata...')

try:
    # Utforska först vilka tabeller som finns under Hushållens ekonomi
    tables = scb_get('HE/HE0110/HE0110A')
    print('Tillgängliga tabeller:')
    for t in tables:
        print(f'  {t["id"]}: {t["text"]}')
except Exception as e:
    print(f'Fel: {e}')

Hämtar inkomstdata...
Tillgängliga tabeller:
  SamForvInk1: Sammanräknad förvärvsinkomst för boende i Sverige hela året efter region, kön, ålder och inkomstklass. År 1999 - 2024
  SamForvInk1c: Sammanräknad förvärvsinkomst för boende i Sverige hela året efter region, utbildningsnivå, kön och ålder. År 2000 - 2024
  SamForvInk1b: Sammanräknad förvärvsinkomst för boende i Sverige hela året efter region, kön och ålder i 1-årsklasser. År 1999 - 2024
  SamForvInk1a: Sammanräknad förvärvsinkomst för boende i Sverige hela året efter kön, inkomstklass och ålder i 1-årsklasser. År 1999 - 2024
  SamForvInk3: Sammanräknad förvärvsinkomst för boende i Sverige hela året efter region, kön, födelseland, vistelsetid i Sverige och ålder. År 2000 - 2024
  SamForvInk2: Sammanräknad förvärvsinkomst för boende i Sverige den 31/12 resp år efter region, kön, ålder och inkomstklass. År 1991 - 2024
  NetInk02: Nettoinkomst för boende i Sverige hela året efter region, kön och ålder. År 2000 - 2024
  NetInk03: N

In [ ]:
# Hitta rätt tabellnamn och hämta inkomstdata
tables = scb_get('HE/HE0110/HE0110A')
print("Tillgängliga tabeller:")
for t in tables:
    print(f"  {t['id']}")

# Testa första tabellen (SamForvInk1)
income_query = {
    'query': [
        {
            'code': 'Region',
            'selection': {
                'filter': 'item',
                'values': [OREBRO_KOMMUN]
            }
        }
    ],
    'response': {'format': 'json'}
}

df_income = scb_post('HE/HE0110/HE0110A/SamForvInk1', income_query)
if not df_income.empty:
    print(f'\nInkomstdata: {len(df_income)} rader')
    display(df_income.tail(10))
    df_income.to_csv('../data/external/scb_inkomst_orebro.csv', index=False)
    print('Sparad!')
else:
    print('Ingen data')

Tillgängliga tabeller:
  SamForvInk1
  SamForvInk1c
  SamForvInk1b
  SamForvInk1a
  SamForvInk3
  SamForvInk2
  NetInk02
  NetInk03
  SamNetFrakt

Inkomstdata: 546 rader


,region,ålder,år,"Medelinkomst, tkr","Medianinkomst, tkr","Totalsumma, mnkr",Antal personer
536,1880,85+,2015,175.5,163.8,630.8,3595
537,1880,85+,2016,181.9,170.7,657.5,3614
538,1880,85+,2017,187.5,176.7,680.0,3626
539,1880,85+,2018,191.4,180.0,679.9,3553
540,1880,85+,2019,197.5,185.7,699.6,3543
541,1880,85+,2020,208.3,195.2,730.5,3506
542,1880,85+,2021,210.9,200.6,746.2,3538
543,1880,85+,2022,221.8,212.4,788.6,3555
544,1880,85+,2023,239.4,225.3,862.4,3603
545,1880,85+,2024,252.8,237.4,955.7,3781


Sparad!


In [ ]:
# Hitta rätt befolkningstabell
tables = scb_get('BE/BE0101/BE0101A')
for t in tables:
    print(f"  {t['id']}: {t['text'][:80]}")

  BefolkManadCKM: Folkmängden per månad efter region, ålder och kön.  År 2025M01 - 2026M01
  BefolkManad: Folkmängden per månad efter region, ålder och kön. År 2000M01 - 2024M12
  BefolkningR1860NCKM: Folkmängden efter ålder och kön.  År 2025
  BefolkningR1860N: Folkmängden efter ålder och kön. År 1860 - 2024
  BefolkningCKM: Folkmängden efter region, civilstånd, ålder och kön. År 2025
  BefolkningNy: Folkmängden efter region, civilstånd, ålder och kön.  År 1968 - 2024
  FolkmangdDecCKM: Folkmängden efter region, ålder (utökade åldersgrupper) och kön. År 2025
  FolkmangdDistrikt: Folkmängden per distrikt, landskap, landsdel eller riket efter kön. År 2015 - 20
  FkvotHVD: Demografisk försörjningskvot efter region. År 2000 - 2024
  FolkmangdNovCKM: Folkmängden den 1 november efter region, ålder och kön.  År 2025
  FolkmangdNov: Folkmängden den 1 november efter region, ålder och kön. År 2002 - 2024


In [ ]:
# 2b. BEFOLKNING per år — Örebro kommun
# Befolkningsdata — enklare tabell (per månad)
df_pop = scb_post('BE/BE0101/BE0101A/BefolkManad', pop_query)
if not df_pop.empty:
    print(f'Befolkningsdata: {len(df_pop)} rader')
    display(df_pop.tail(10))
    df_pop.to_csv('../data/external/scb_befolkning_orebro.csv', index=False)
    print('Sparad!')
else:
    print('Ingen data')

Befolkningsdata: 300 rader


,region,månad,Antal
290,1880,2024M03,159408
291,1880,2024M04,159382
292,1880,2024M05,159310
293,1880,2024M06,159187
294,1880,2024M07,159300
295,1880,2024M08,159804
296,1880,2024M09,160070
297,1880,2024M10,160143
298,1880,2024M11,160158
299,1880,2024M12,160140


Sparad!


In [ ]:
# 2c. FASTIGHETSPRISER — genomsnittspriser per kommun 

# Utforska bostadstabeller
print('Utforskar bostadspristabeller...')

try:
    bo_tables = scb_get('BO/BO0501/BO0501D')
    for t in bo_tables:
        print(f'  {t["id"]}: {t["text"]}')
except Exception as e:
    print(f'Fel: {e}')
    print('Testar alternativ sökväg...')
    try:
        bo_tables = scb_get('BO/BO0501')
        for t in bo_tables:
            print(f'  {t["id"]}: {t["text"]}')
    except Exception as e2:
        print(f'Fel igen: {e2}')

Utforskar bostadspristabeller...
  LagfartAllaRegionAr: Beviljade lagfarter för samtliga fastighetstyper efter region (län, riket) och typ av överlåtelse. År 1999 - 2024


In [ ]:
 # Hämta fastighetspriser — lagfarter per region och år
price_query = {
    'query': [
        {
            'code': 'Region',
            'selection': {
                'filter': 'item',
                'values': ['18']  # Örebro län
            }
        }
    ],
    'response': {'format': 'json'}
}

df_prices = scb_post('BO/BO0501/BO0501D/LagfartAllaRegionAr', price_query)
if not df_prices.empty:
    print(f'Fastighetsprisdata: {len(df_prices)} rader')
    display(df_prices.tail(10))
    df_prices.to_csv('../data/external/scb_fastighetspriser_orebro.csv', index=False)
    print('Sparad!')
else:
    print('Ingen data')

Fastighetsprisdata: 286 rader


,region,typ av överlåtelse,år,Antal,Summa köpeskilling i tkr
276,18,90,2015,19,0
277,18,90,2016,10,0
278,18,90,2017,13,0
279,18,90,2018,26,0
280,18,90,2019,14,0
281,18,90,2020,10,0
282,18,90,2021,18,0
283,18,90,2022,21,0
284,18,90,2023,13,0
285,18,90,2024,6,0


Sparad!


In [ ]:
## 3. Snabb kvalitetskoll
# Ladda den sparade datan
import glob

df = pd.read_csv('../data/raw/hemnet_orebro_fixed.csv')

if csv_files:
    latest = sorted(csv_files)[-1]
    df = pd.read_csv(latest)
    
    print(f'Laddad: {latest}')
    print(f'Rader: {len(df)}')
    print(f'Kolumner: {list(df.columns)}')
    
    print(f'\n--- Datatyper ---')
    print(df.dtypes)
    
    print(f'\n--- Saknade värden ---')
    missing = df.isnull().sum()
    print(missing[missing > 0])
    
    print(f'\n--- Grundstatistik ---')
    display(df.describe())
    
    print(f'\n--- Bostadstyper ---')
    if 'bostadstyp' in df.columns:
        print(df['bostadstyp'].value_counts())
    
    print(f'\n--- Stickprov (5 rader) ---')
    display(df.sample(5))
else:
    print('Ingen sparad CSV hittad i ../data/raw/')
    print('Kör scraping-cellerna ovan först.')

Laddad: ../data/raw/hemnet_orebro_fixed.csv
Rader: 6622
Kolumner: ['slutpris', 'avgift_kr', 'boarea_kvm', 'antal_rum', 'sald_datum', 'prisforandring_pct', 'pris_per_kvm', 'omrade', 'url', 'bostadstyp', 'raw_text']

--- Datatyper ---
slutpris                int64
avgift_kr             float64
boarea_kvm            float64
antal_rum             float64
sald_datum                str
prisforandring_pct    float64
pris_per_kvm          float64
omrade                    str
url                       str
bostadstyp                str
raw_text                  str
dtype: object

--- Saknade värden ---
avgift_kr             2864
boarea_kvm               5
antal_rum              108
prisforandring_pct      40
pris_per_kvm          2903
omrade                  30
dtype: int64

--- Grundstatistik ---


,slutpris,avgift_kr,boarea_kvm,antal_rum,prisforandring_pct,pris_per_kvm
count,6.622000e+03,3758.000000,6617.000000,6514.000000,6582.000000,3719.000000
mean,2.796085e+06,4836.742416,104.700559,4.017578,2.134458,25107.589944
std,1.557041e+06,1339.633745,52.900187,1.608301,11.834458,7691.586238
min,1.100000e+05,556.000000,15.000000,1.000000,-86.000000,3947.000000
25%,1.670000e+06,3971.250000,75.000000,3.000000,-4.000000,19685.000000
50%,2.407500e+06,4817.000000,100.000000,4.000000,0.000000,24630.000000
75%,3.550000e+06,5651.000000,124.000000,5.000000,6.000000,30000.000000
max,1.400000e+07,11720.000000,990.000000,13.000000,128.000000,72674.000000



--- Bostadstyper ---
bostadstyp
lagenheter    2500
villor        2497
radhus        1625
Name: count, dtype: int64

--- Stickprov (5 rader) ---


,slutpris,avgift_kr,boarea_kvm,antal_rum,sald_datum,prisforandring_pct,pris_per_kvm,omrade,url,bostadstyp,raw_text
5165,1900000,5955.0,79.0,3.0,2025-04-11,6.0,24051.0,G Radhus Nya Hjärsta,https://www.hemnet.se/salda/radhus-3rum-nya-hj...,radhus,Såld 11 apr. 2025 Skiftesvägen 2G Radhus Nya H...
5614,2900000,4155.0,113.0,5.0,2022-07-27,14.0,25664.0,A Radhus Vintrosa,https://www.hemnet.se/salda/radhus-5rum-vintro...,radhus,Såld 27 jul. 2022 Granlidsvägen 5A Radhus Vint...
4533,6900000,NaN,260.0,8.0,2021-01-26,22.0,NaN,B Villa Brickeberg,https://www.hemnet.se/salda/villa-8rum-brickeb...,villor,Såld 26 jan. 2021 Brickebergsvägen 21B Villa B...
3231,3025000,NaN,99.0,4.0,2024-05-11,16.0,NaN,Villa Mosås,https://www.hemnet.se/salda/villa-4rum-mosas-o...,villor,Maps Data: Google © 2026 Såld 11 maj. 2024 Met...
1450,1265000,1936.0,32.0,1.0,2024-02-17,-2.0,39531.0,Lägenhet Vasastan,https://www.hemnet.se/salda/lagenhet-1rum-vasa...,lagenheter,Maps Data: Google © 2026 Såld 17 feb. 2024 Her...
